# J-space battery

Turns the J-space swap claim into rates over the paper's own two-hop battery (`data/probe-swap.json`, from `anthropics/jacobian-lens`, Apache-2.0). For each item and each fitted lens layer $\ell$ it measures:

- **baseline:** does a greedy continuation start with the answer? Only baseline-correct items can test a swap.
- **readout:** the best lens rank of the intermediate and of the answer over the prompt positions at $\ell$.
- **intermediate swap:** patch the (intermediate, swap-to) J-space pair at every prompt position of $\ell$ and score whether the swapped answer becomes top-1, or enters the top-$k$ on items where it was not already there.
- **answer swap:** the same with the (answer, swapped answer) pair.
- **control swap:** the intermediate swapped for an unrelated entity, scored on the same swapped answer. This is the specificity check every swap rate is read against.

Every write uses interpretune's production path: `resolve_jlens_layer`, `jlens_direction_rows` in the norm-aware basis, and the `patch` intervention at all positions through the NNsight model backend.

In [ ]:
EXPERIMENT_CONFIG_PATH = "configs/gemma3_1b_it_probe_swap_smoke.yaml"
EXPERIMENT_CONFIG_NAME = "gemma3_1b_it_probe_swap_smoke"
EXPERIMENT_NAME = "gemma3_1b_it_probe_swap_smoke"
VERBOSE = True

In [ ]:
import json
import os
import sys
from pathlib import Path

SNAPSHOT_ROOT = Path.cwd().parent  # the component root: data/, exp/, jspace_battery/
sys.path.insert(0, str(SNAPSHOT_ROOT))

from interpretune.harness import load_experiment_config
from exp.jspace_battery import battery_config_from_experiment, rate_table, run_battery

In [ ]:
cfg = load_experiment_config(EXPERIMENT_CONFIG_PATH)
if cfg["EXPERIMENT_NAME"] != EXPERIMENT_NAME:
    raise ValueError(f"config EXPERIMENT_NAME {cfg['EXPERIMENT_NAME']!r} != parameter {EXPERIMENT_NAME!r}")
battery_cfg = battery_config_from_experiment(cfg)
battery_cfg

In [ ]:
result = run_battery(battery_cfg, progress=print if VERBOSE else None)
summary = result["summary"]
{k: v for k, v in summary.items() if not isinstance(v, dict)}

## Rates per layer

Rates are over baseline-correct items whose entities are single tokens (`n_usable`). The top-$k$ swap rates count only discriminating items, whose swapped answer was not already in the baseline top-$k$.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(rate_table(summary, result["lens"]["source_layers"] if battery_cfg.layers is None else battery_cfg.layers)))

In [ ]:
bands = {k: v for k, v in summary.items() if "_band_" in k}
bands

In [ ]:
out_dir = Path(cfg.get("RUNTIME", {}).get("experiment_work_dir") or "outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{EXPERIMENT_NAME}.json"
out_path.write_text(json.dumps(result, default=str))
print(out_path)